In [1]:
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [2]:
# 读取数据集
df = pd.read_excel('训练数据_校准后_窗口 6.xlsx')

# 提取特征和目标变量
X = df[['钻压(kN)', '瞬时机械钻速(m/h)', '立管压力(MPa)']]
y = df['是否存在裂缝']

# 对目标变量进行编码
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 数据标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 使用 SMOTE 进行过采样
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_scaled, y_encoded)

# 划分训练集和测试集
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)

In [3]:
# 高斯过程分类器
kernel = 1.0 * RBF(1.0)
gpc = GaussianProcessClassifier(kernel=kernel, random_state=42)

# 随机森林分类器
rf = RandomForestClassifier(random_state=42)

# 定义参数网格
param_grid_gpc = {
    'kernel': [1.0 * RBF(1.0), 1.0 * RBF(10.0)]
}

param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30]
}

# 使用网格搜索进行参数调优
grid_search_gpc = GridSearchCV(gpc, param_grid_gpc, cv=3)
grid_search_rf = GridSearchCV(rf, param_grid_rf, cv=3)

# 训练模型
grid_search_gpc.fit(X_train, y_train)
grid_search_rf.fit(X_train, y_train)

# 获取最优模型
best_gpc = grid_search_gpc.best_estimator_
best_rf = grid_search_rf.best_estimator_

# 在测试集上进行预测
y_pred_gpc = best_gpc.predict(X_test)
y_pred_rf = best_rf.predict(X_test)

# 将编码后的预测结果转换回原始标签
y_pred_gpc_labels = le.inverse_transform(y_pred_gpc)
y_pred_rf_labels = le.inverse_transform(y_pred_rf)
y_test_labels = le.inverse_transform(y_test)


In [4]:
# 评估高斯过程分类器
accuracy_gpc = accuracy_score(y_test_labels, y_pred_gpc_labels)
precision_gpc = precision_score(y_test_labels, y_pred_gpc_labels, pos_label='有')
recall_gpc = recall_score(y_test_labels, y_pred_gpc_labels, pos_label='有')
f1_gpc = f1_score(y_test_labels, y_pred_gpc_labels, pos_label='有')

# 评估随机森林分类器
accuracy_rf = accuracy_score(y_test_labels, y_pred_rf_labels)
precision_rf = precision_score(y_test_labels, y_pred_rf_labels, pos_label='有')
recall_rf = recall_score(y_test_labels, y_pred_rf_labels, pos_label='有')
f1_rf = f1_score(y_test_labels, y_pred_rf_labels, pos_label='有')

print("高斯过程分类器评估结果：")
print(f"  准确率: {accuracy_gpc}")
print(f"  正类精确率: {precision_gpc}")
print(f"  正类召回率: {recall_gpc}")
print(f"  正类 F1 分数: {f1_gpc}")

print("随机森林分类器评估结果：")
print(f"  准确率: {accuracy_rf}")
print(f"  正类精确率: {precision_rf}")
print(f"  正类召回率: {recall_rf}")
print(f"  正类 F1 分数: {f1_rf}")
    

高斯过程分类器评估结果：
  准确率: 0.8783253379851723
  正类精确率: 0.840568271507498
  正类召回率: 0.9325744308231173
  正类 F1 分数: 0.8841843088418431
随机森林分类器评估结果：
  准确率: 0.935455734845181
  正类精确率: 0.9406028368794326
  正类召回率: 0.9290718038528897
  正类 F1 分数: 0.9348017621145375
